# How a beam spreads a point load through an elastic support

Imagine looking under a long workbench rail carried on an elastic strip. Guess which part of the strip supports half the load. Compute the deflection and the cumulative reaction below. The default parameters describe a bilateral linear support. Change the four numbers in the first cell and run the cells in order.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

E = 70e9             # Young modulus, Pa
width, depth = 0.04, 0.02  # rectangular rail, m
k = 1.5e6            # foundation force per length per displacement, N/m²
P = 150.0            # downward point load, N
assert min(E, width, depth, k, P) > 0

## Construct the field from the half-line solution
The support reaction per unit length is $k w$. Symmetry makes the slope zero at the load. The jump in the third derivative supplies the concentrated force. The cumulative signed reaction within a symmetric interval follows by integrating the resulting field.

In [ ]:
I = width * depth**3 / 12
EI = E * I
beta = (k / (4*EI))**0.25
length_scale = 1/beta
w0 = P*beta/(2*k)

def deflection(x):
    s = beta * np.abs(x)
    return w0*np.exp(-s)*(np.cos(s) + np.sin(s))

def captured(distance):
    s = beta * np.abs(distance)
    return 1 - np.exp(-s)*np.cos(s)

print(f"EI = {EI:.3f} N m²; decay length = {length_scale:.4f} m")
print(f"Centre deflection = {w0*1e3:.4f} mm")
print(f"Fraction within one decay length = {captured(length_scale):.5f}")

## Solve an independently discretised beam
Represent the curvature by second differences on a long simply supported interval. Form the energy Hessian and solve for the nodal displacement. The middle force is divided by the grid spacing to express it as a load per length. The cells below run with NumPy; a local OneSimulation installation can supply the same linear solve.

In [ ]:
try:
    from onesim.numerics.linear.matrices.dense_linear_algebra import solve_dense
    backend = "OneSimulation dense linear solver"
except ImportError:
    solve_dense = np.linalg.solve
    backend = "NumPy linear solver"

def discrete_beam(points):
    edge = 8*length_scale
    x = np.linspace(-edge, edge, points+2)[1:-1]
    h = x[1]-x[0]
    D2 = (-2*np.eye(points) + np.eye(points,k=1) + np.eye(points,k=-1))/h**2
    force = np.zeros(points)
    force[points//2] = P/h
    w = solve_dense(EI*(D2.T @ D2) + k*np.eye(points), force)
    return x, w

for count in (65,129):
    x_grid, w_grid = discrete_beam(count)
    discrepancy = abs(w_grid[count//2]/w0-1)
    print(f"{backend}, {count} unknowns: centre error {100*discrepancy:.3f}%")
assert discrepancy < 0.015
assert abs(captured(12*length_scale)-1) < 1e-5

## Observe the field and the signed support force
The bending moment has a different shape from the displacement. The right panel integrates reactions outward from the load. The reference at one marks the full applied force.

In [ ]:
x = np.linspace(-4*length_scale,4*length_scale,501)
d = np.linspace(0,5*length_scale,320)
fig, ax = plt.subplots(1,2,figsize=(9,3.5),layout="constrained")
ax[0].plot(x,1e3*deflection(x),label="Infinite-beam solution")
mask = np.abs(x_grid) <= 4*length_scale
ax[0].plot(x_grid[mask][::2],1e3*w_grid[mask][::2],"o",ms=3,label="Discrete energy solution")
ax[0].axhline(0,color="0.5",lw=0.8)
ax[0].set(xlabel="Rail coordinate (m)",ylabel="Downward deflection (mm)",title="Local rail displacement")
ax[1].plot(d,captured(d),label="Signed reaction / load")
ax[1].axhline(1,color="0.45",ls=":",label="Applied load")
ax[1].set(xlabel="Distance on each side (m)",ylabel="Fraction of load",title="Reaction collected under the rail")
for a in ax:
    a.grid(alpha=.22); a.legend(fontsize=8)
plt.show()

## Change the experiment
Double the rail depth before rerunning. Compare the ratio of decay lengths to $2^{3/4}$. Hold the section fixed and increase the foundation stiffness sixteenfold. Explain any values above one in the cumulative reaction. Which part of the plotted deflection would make a support that can only push lose contact?